In [1]:
import pymongo
import requests
import time
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed

# ============================================================
# CONFIGURATION
# ============================================================
BATCH_LIMIT = None        # None = process everything remaining; or set a number to cap this run
MAX_WORKERS = 5            # concurrent requests - keep modest, this is a public gov't API
REQUEST_TIMEOUT = 15       # seconds before giving up on a single request
PROGRESS_EVERY = 100       # print a progress line every N companies processed

BASE_URL = "https://data.brreg.no/regnskapsregisteret/regnskap/"
MONGO_URI = "mongodb://mongodb:27017/"

# ============================================================
# SETUP
# ============================================================
client = pymongo.MongoClient(MONGO_URI)
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# ============================================================
# STEP 1: Figure out what's left to fetch
# ============================================================
print("Loading all organisasjonsnummer from companies collection...")
all_org_numbers = set(
    doc["organisasjonsnummer"]
    for doc in companies_col.find({}, {"organisasjonsnummer": 1, "_id": 0})
)
print(f"Total companies: {len(all_org_numbers):,}")

print("Loading already-fetched organisasjonsnummer from financial_data collection...")
already_fetched = set(
    doc["_id"]
    for doc in financial_col.find({}, {"_id": 1})
)
print(f"Already fetched: {len(already_fetched):,}")

remaining = list(all_org_numbers - already_fetched)
print(f"Remaining before this run: {len(remaining):,}")

if BATCH_LIMIT is not None:
    remaining = remaining[:BATCH_LIMIT]
    print(f"BATCH_LIMIT set - processing {len(remaining):,} this run")
else:
    print(f"No BATCH_LIMIT - will process all {len(remaining):,} remaining (stop anytime with the Jupyter Stop button)")
print()

# ============================================================
# STEP 2: Fetch function for a single company
# ============================================================
def fetch_financial_data(org_nr):
    """
    Fetches financial data for one organisasjonsnummer.
    Returns a dict for DEFINITIVE outcomes (success or confirmed no-data).
    Returns None for transient problems, leaving that company unfetched
    so it is automatically retried on a future run.
    """
    url = f"{BASE_URL}{org_nr}"
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
    except requests.exceptions.RequestException as e:
        return None  # network-level failure - transient, retry later

    if response.status_code == 200:
        return {
            "_id": org_nr,
            "organisasjonsnummer": org_nr,
            "fetch_status": "success",
            "http_status": 200,
            "fetched_at": datetime.now(timezone.utc),
            "data": response.json()
        }
    elif response.status_code == 404:
        # Confirmed: no accounts filed for this company - a real, final outcome
        return {
            "_id": org_nr,
            "organisasjonsnummer": org_nr,
            "fetch_status": "no_data",
            "http_status": 404,
            "fetched_at": datetime.now(timezone.utc),
            "data": None
        }
    else:
        # 429 (rate limited) or any other unexpected status - transient, retry later
        return None

def save_result(result):
    """
    Writes one result to MongoDB. Wrapped in try/except so a temporary
    MongoDB outage doesn't crash the whole run - it just skips this
    write, and the company stays eligible for retry next time.
    """
    try:
        financial_col.replace_one({"_id": result["_id"]}, result, upsert=True)
        return True
    except pymongo.errors.PyMongoError as e:
        print(f"  [MONGO WRITE FAILED] {result['_id']}: {e}")
        return False

# ============================================================
# STEP 3: Run, with graceful stop support
# ============================================================
success_count = 0
no_data_count = 0
skipped_count = 0
processed_count = 0
start_time = time.time()

executor = ThreadPoolExecutor(max_workers=MAX_WORKERS)
futures = {executor.submit(fetch_financial_data, org_nr): org_nr for org_nr in remaining}

try:
    for future in as_completed(futures):
        result = future.result()
        processed_count += 1

        if result is not None:
            if save_result(result):
                if result["fetch_status"] == "success":
                    success_count += 1
                else:
                    no_data_count += 1
            else:
                skipped_count += 1  # write failed, will retry next run
        else:
            skipped_count += 1  # fetch failed, will retry next run

        if processed_count % PROGRESS_EVERY == 0:
            elapsed = time.time() - start_time
            rate = processed_count / elapsed if elapsed > 0 else 0
            remaining_this_run = len(remaining) - processed_count
            eta_minutes = (remaining_this_run / rate / 60) if rate > 0 else float("inf")
            print(f"[{processed_count:,}/{len(remaining):,}] "
                  f"success={success_count:,} no_data={no_data_count:,} skipped={skipped_count:,} "
                  f"| {rate:.1f} req/s | ETA {eta_minutes:.1f} min")

    executor.shutdown(wait=True)

except KeyboardInterrupt:
    # Triggered by clicking the Jupyter Stop button
    print("\n\n=== STOP REQUESTED ===")
    print("Cancelling queued requests, waiting for in-flight ones to finish...")
    executor.shutdown(wait=True, cancel_futures=True)
    print("Stopped cleanly. All completed work is saved in MongoDB.")
    print("Re-run this cell anytime to continue from where you left off.\n")

# ============================================================
# SUMMARY
# ============================================================
elapsed_total = time.time() - start_time
total_fetched_now = financial_col.count_documents({})
print(f"\n=== RUN SUMMARY ===")
print(f"Processed this run:  {processed_count:,}")
print(f"  Success (200):     {success_count:,}")
print(f"  No data (404):     {no_data_count:,}")
print(f"  Skipped (retry):   {skipped_count:,}")
print(f"Time elapsed:        {elapsed_total/60:.1f} minutes")
print(f"Total progress:      {total_fetched_now:,} / {len(all_org_numbers):,} "
      f"({total_fetched_now/len(all_org_numbers)*100:.1f}%)")

Loading all organisasjonsnummer from companies collection...
Total companies: 1,171,373
Loading already-fetched organisasjonsnummer from financial_data collection...
Already fetched: 1,170,291
Remaining before this run: 1,082
No BATCH_LIMIT - will process all 1,082 remaining (stop anytime with the Jupyter Stop button)

[100/1,082] success=0 no_data=0 skipped=100 | 26.5 req/s | ETA 0.6 min
[200/1,082] success=0 no_data=0 skipped=200 | 27.9 req/s | ETA 0.5 min
[300/1,082] success=0 no_data=0 skipped=300 | 28.2 req/s | ETA 0.5 min
[400/1,082] success=0 no_data=0 skipped=400 | 27.4 req/s | ETA 0.4 min
[500/1,082] success=0 no_data=0 skipped=500 | 28.1 req/s | ETA 0.3 min
[600/1,082] success=0 no_data=0 skipped=600 | 28.6 req/s | ETA 0.3 min
[700/1,082] success=1 no_data=0 skipped=699 | 28.6 req/s | ETA 0.2 min
[800/1,082] success=1 no_data=0 skipped=799 | 28.8 req/s | ETA 0.2 min
[900/1,082] success=1 no_data=0 skipped=899 | 29.1 req/s | ETA 0.1 min
[1,000/1,082] success=1 no_data=0 skippe

In [2]:
import pymongo
import requests
import time
from collections import Counter

BASE_URL = "https://data.brreg.no/regnskapsregisteret/regnskap/"
MONGO_URI = "mongodb://mongodb:27017/"

client = pymongo.MongoClient(MONGO_URI)
db = client["companiesdb"]
companies_col = db["companies"]
financial_col = db["financial_data"]

# Recompute the outstanding set exactly as the fetch cell does.
all_org = set(
    d["organisasjonsnummer"]
    for d in companies_col.find({}, {"organisasjonsnummer": 1, "_id": 0})
)
done = set(d["_id"] for d in financial_col.find({}, {"_id": 1}))
remaining = sorted(all_org - done)
print("Remaining: %d\n" % len(remaining))

# Sample sequentially with a deliberate 1-second gap. If these still fail at
# 1 req/s, concurrency is not the cause.
statuses = Counter()
for org in remaining[:20]:
    try:
        r = requests.get(BASE_URL + org, timeout=15)
    except requests.exceptions.RequestException as e:
        statuses["EXCEPTION: %s" % type(e).__name__] += 1
        print("%s  EXCEPTION  %s" % (org, e))
        time.sleep(1.0)
        continue

    statuses[r.status_code] += 1
    retry_after = r.headers.get("Retry-After", "-")
    body = r.text[:150].replace("\n", " ")
    print("%s  HTTP %s  Retry-After=%s  %s" % (org, r.status_code, retry_after, body))
    time.sleep(1.0)

print("\nStatus distribution over the sample:")
for k, v in statuses.most_common():
    print("  %s: %d" % (k, v))

Remaining: 1082

812966022  HTTP 500  Retry-After=-  {"timestamp":"2026-09-02T18:09:14.542+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
813062402  HTTP 500  Retry-After=-  {"timestamp":"2026-09-02T18:09:15.687+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
813918102  HTTP 500  Retry-After=-  {"timestamp":"2026-09-02T18:09:16.852+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
814115232  HTTP 500  Retry-After=-  {"timestamp":"2026-09-02T18:09:18.003+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
815611942  HTTP 500  Retry-After=-  {"timestamp":"2026-09-02T18:09:19.152+0000","status":"500","error":"Internal Server Error","message":"An error occurred while processing the request."
816521432  HTTP 500  Retry-After=-  {"timestamp"

In [3]:
sample_ids = remaining[:1000]
pipeline = [
    {"$match": {"organisasjonsnummer": {"$in": sample_ids}}},
    {"$group": {"_id": "$organisasjonsform.kode", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
]
for row in companies_col.aggregate(pipeline):
    print("%-8s %d" % (row["_id"], row["n"]))

VPFO     250
STI      200
FLI      148
AS       114
PK       69
SPA      61
NUF      53
ENK      31
DA       25
GFS      23
ASA      16
ANS      5
SA       2
BA       1
SÆR      1
ESEK     1


In [4]:
import random

random.seed(42)
sample = random.sample(remaining, 30)

statuses = Counter()
for org in sample:
    try:
        r = requests.get(BASE_URL + org, timeout=15)
        statuses[r.status_code] += 1
    except requests.exceptions.RequestException as e:
        statuses["EXCEPTION: %s" % type(e).__name__] += 1
    time.sleep(1.0)

print("Random sample of 30 across the full remaining set:")
for k, v in statuses.most_common():
    print("  %s: %d" % (k, v))

# What kind of entities are these? You never posted this from the last cell.
pipeline = [
    {"$match": {"organisasjonsnummer": {"$in": remaining}}},
    {"$group": {"_id": "$organisasjonsform.kode", "n": {"$sum": 1}}},
    {"$sort": {"n": -1}},
]
print("\nLegal form breakdown of all %d remaining:" % len(remaining))
for row in companies_col.aggregate(pipeline):
    print("  %-8s %d" % (row["_id"], row["n"]))

Random sample of 30 across the full remaining set:
  500: 30

Legal form breakdown of all 1082 remaining:
  VPFO     271
  STI      219
  FLI      159
  AS       129
  PK       71
  SPA      61
  NUF      56
  ENK      33
  DA       31
  GFS      23
  ASA      18
  ANS      5
  SA       3
  SÆR      1
  ESEK     1
  BA       1
